<a href="https://colab.research.google.com/github/caniletgo/ML/blob/main/%D0%9B%D0%A09_%D0%A7%D0%B5%D1%80%D0%BD%D0%B8%D1%88_%D0%A4%D0%86%D0%A23_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Черниш Ірина Віталіївна  
**Група:** ФІТ 3-10
**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
print("Унікальних користувачів:", ratings_df['user_id'].nunique())
print("Унікальних фільмів:", ratings_df['item_id'].nunique())
print("\nРозподіл оцінок:\n", ratings_df['rating'].value_counts().sort_index())

Унікальних користувачів: 943
Унікальних фільмів: 1682

Розподіл оцінок:
 rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

**Короткі спостереження щодо даних:**
Набір даних `ml-100k` містить 100 000 оцінок від 943 унікальних користувачів для 1682 фільмів[cite: 65]. Оцінки виставлені за цілочисельною шкалою від 1.0 до 5.0 балів. Більшість оцінок є позитивними (найчастіше користувачі ставлять 4.0 та 3.0 бали, середній рейтинг становить близько 3.53).

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [6]:
reader = Reader(
    rating_scale=data.reader.rating_scale
)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', len(testset))

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

**Обґрунтування вибору параметрів:**
- **Для моделі SVD (матрична факторизація):**
  - `n_factors` (`[50, 100]`) — кількість прихованих (латентних) факторів, що описують вподобання користувачів та характеристики фільмів; дозволяє перевірити баланс між виразністю моделі та ризиком перенавчання.
  - `lr_all` (`[0.005, 0.010]`) — швидкість навчання стохастичного градієнтного спуску (стандартне значення 0.005 та прискорене 0.010 для швидшої збіжності за 20 епох).
  - `reg_all` (`[0.02, 0.05]`) — коефіцієнт L2-регуляризації, який стримує зростання ваг і запобігає перенавчанню на розрідженій матриці оцінок.
- **Для моделі KNNBasic (колаборативна фільтрація на основі сусідів):**
  - `k` (`[20, 40]`) — максимальна кількість найближчих сусідів, що враховується під час прогнозування рейтингу.
  - `min_k` (`[1, 5]`) — мінімальна кількість сусідів, необхідна для побудови прогнозу (збільшення до 5 відсіює ненадійні оцінки за одиничними збігами).
  - `sim_options` (`name`: `['cosine', 'pearson']`, `user_based`: `[False, True]`) — вибір метрики схожості (косинусна відстань або кореляція Пірсона) та підходу до пошуку сусідів: між фільмами (`item-based`, `False`) або між користувачами (`user-based`, `True`).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [10]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [11]:
print('\nНайкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nНайкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])


Найкращий результат SVD:
RMSE: 0.9425566798054708
MAE: 0.7438060178310973
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Найкращий результат KNNBasic:
RMSE: 1.027641796636343
MAE: 0.8147708999862369
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
| :--- | :---: | :---: | :--- |
| SVD | 0.9426 | 0.7438 | `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}` |
| KNNBasic | 1.0276 | 0.8148 | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

На перехресній перевірці (`cv=3`) кращу якість продемонструвала модель **SVD** (RMSE = 0.9426, MAE = 0.7438), суттєво випередивши **KNNBasic** (RMSE = 1.0276, MAE = 0.8148). Для `SVD` оптимальними стали параметри `n_factors=50`, `lr_all=0.01`, `reg_all=0.05`, а для `KNNBasic` — колаборативна фільтрація між користувачами (`user_based=True`) з косинусною подібністю (`cosine`), `k=40` та `min_k=1`.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [12]:
# Отримуємо найкращі параметри за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# Формуємо повну навчальну вибірку
trainset = train_data.build_full_trainset()

# Навчаємо найкращу модель SVD
best_svd = SVD(
    **best_svd_params,
    random_state=42
)
best_svd.fit(trainset)

# Навчаємо найкращу модель KNNBasic
best_knn = KNNBasic(
    **best_knn_params
)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [13]:
# Отримуємо прогнози на незалежній тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# Оцінюємо SVD
print('\nРезультати SVD на тестовій вибірці:')
svd_rmse = accuracy.rmse(
    svd_predictions,
    verbose=True
)
svd_mae = accuracy.mae(
    svd_predictions,
    verbose=True
)

# Оцінюємо KNNBasic
print('\nРезультати KNNBasic на тестовій вибірці:')
knn_rmse = accuracy.rmse(
    knn_predictions,
    verbose=True
)
knn_mae = accuracy.mae(
    knn_predictions,
    verbose=True
)


Результати SVD на тестовій вибірці:
RMSE: 0.9192
MAE:  0.7234

Результати KNNBasic на тестовій вибірці:
RMSE: 1.0120
MAE:  0.7984


In [14]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

print('\nПорівняння моделей:')
display(results)


Порівняння моделей:


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
| :--- | :---: | :---: |
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** Оцінки на незалежній тестовій вибірці узгоджуються з результатами перехресної перевірки та суттєво від них не відрізняються: для **SVD** похибка RMSE знизилася з 0.9426 на CV до 0.9192 на тесті (MAE — з 0.7438 до 0.7234), а для **KNNBasic** — з 1.0276 до 1.0120 (MAE — з 0.8148 до 0.7984)[cite: 52, 54]. Незначне покращення метрик на тестовій вибірці пояснюється тим, що фінальні моделі навчалися на 100% навчального набору (`build_full_trainset()`), а не на 2/3 даних, як під час 3-фолдової кросвалідації[cite: 48, 50]. Це підтверджує відсутність перенавчання та стабільну перевагу моделі **SVD**[cite: 54].

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Для рекомендацій використовується модель: {best_model_name}")

# TODO 8.2 — Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)

    # TODO 8.3 — Знайдіть неоцінені ним фільми.
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    rated_raw_iids = {trainset.to_raw_iid(iid) for iid in rated_inner_iids}

    print(f"Кількість оцінених фільмів користувачем {user_id}: {len(rated_raw_iids)}")

    all_raw_iids = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]
    unrated_iids = [iid for iid in all_raw_iids if iid not in rated_raw_iids]

    # TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
    predictions = [
        (iid, best_model.predict(uid=user_id, iid=iid).est)
        for iid in unrated_iids
    ]
    predictions.sort(key=lambda x: x[1], reverse=True)

    recommendations_df = pd.DataFrame(
        predictions[:10],
        columns=['item_id', 'predicted_rating']
    )

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(recommendations_df)

except ValueError:
    print(
        f"Користувача з ID {user_id} "
        "немає в навчальній вибірці."
    )

Для рекомендацій використовується модель: SVD
Кількість оцінених фільмів користувачем 196: 29

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Висновок:**
За результатами перехресної перевірки (`cv=3`) найкращою моделлю обрано **SVD**, оскільки вона продемонструвала суттєво нижчу похибку прогнозування (`CV RMSE = 0.9426`, `CV MAE = 0.7438`) порівняно з **KNNBasic** (`CV RMSE = 1.0276`, `CV MAE = 0.8148`). Для моделі **SVD** оптимальними гіперпараметрами стали `n_factors = 50`, `lr_all = 0.01` та `reg_all = 0.05`, тоді як для **KNNBasic** найкращий результат забезпечили `k = 40`, `min_k = 1` та підхід `user_based = True` із косинусною метрикою подібності (`cosine`). На незалежній тестовій вибірці модель **SVD** підтвердила свою перевагу, показавши підсумкові значення `Test RMSE = 0.9192` та `Test MAE = 0.7234`, тоді як **KNNBasic** отримала `Test RMSE = 1.0120` та `Test MAE = 0.7984`. Різниця між результатами кросвалідації та фінального тестування є незначною, а дещо нижчі похибки на тесті пояснюються навчанням остаточних моделей на повному навчальному наборі (`80%` усіх даних замість `2/3` на кожному фолді CV), що свідчить про відсутність перенавчання. На основі найкращої моделі **SVD** для користувача `196` (який оцінив 29 фільмів у навчальній вибірці) було сформовано персоналізований список топ-10 рекомендацій із виключенням уже переглянутих стрічок. Основним обмеженням побудованих колаборативних моделей є проблема «холодного старту» (cold start): алгоритм не може сформувати персоналізовані рекомендації для нових користувачів або коректно оцінити нові фільми, які відсутні у навчальній вибірці `trainset`.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [16]:
import numpy as np

# Створюємо матрицю "користувач - фільм" на навчальних даних
user_item_matrix = train_df.pivot(index='user_id', columns='item_id', values='rating')
user_means = user_item_matrix.mean(axis=1)
matrix_centered = user_item_matrix.sub(user_means, axis=0).fillna(0)

# Косинусна подібність між користувачами (центрирована = кореляція Пірсона)
norms = np.linalg.norm(matrix_centered.values, axis=1, keepdims=True)
norms[norms == 0] = 1e-9
norm_matrix = matrix_centered.values / norms
user_sim = pd.DataFrame(
    norm_matrix @ norm_matrix.T,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

def predict_rating_custom(uid, iid, k=30):
    if uid not in user_item_matrix.index or iid not in user_item_matrix.columns:
        return train_df['rating'].mean()
    # Користувачі, які оцінили цей фільм
    raters = user_item_matrix[iid].dropna().index
    if len(raters) == 0:
        return user_means[uid]
    sims = user_sim.loc[uid, raters].nlargest(k)
    sims = sims[sims > 0]
    if sims.empty:
        return user_means[uid]
    diffs = matrix_centered.loc[sims.index, iid]
    pred = user_means[uid] + np.dot(sims.values, diffs.values) / sims.sum()
    return float(np.clip(pred, 1.0, 5.0))

# Оцінка RMSE власного алгоритму на тестовій вибірці
custom_preds = [predict_rating_custom(u, i) for u, i, _ in testset]
actuals = [r for _, _, r in testset]
custom_rmse = np.sqrt(np.mean((np.array(actuals) - np.array(custom_preds)) ** 2))
print(f"Custom User-Based CF Test RMSE: {custom_rmse:.4f}")

Custom User-Based CF Test RMSE: 0.9353
